In [0]:
%run ../common/config

In [0]:
from pyspark.sql.functions import *

In [0]:
providers_df= spark.read.table(f"{env_catalog}.bronze.providers")

In [0]:
providers_df.printSchema()

In [0]:
bronze_providers = providers_df.dropDuplicates(["Id"])

In [0]:
silver_providers = (
    bronze_providers
    .select(
        col("Id").alias("provider_id"),
        col("ORGANIZATION").alias("organization_id"),
        col("NAME").alias("provider_name"),
        col("GENDER").alias("gender"),
        col("SPECIALITY").alias("speciality"),
        col("ADDRESS").alias("address"),
        col("CITY").alias("city"),
        col("STATE").alias("state"),
        col("ZIP").alias("zip_code"),
        col("LAT").alias("latitude"),
        col("LON").alias("longitude"),
        col("ENCOUNTERS").alias("total_encounters"),
        col("PROCEDURES").alias("total_procedures"),
        "source_file",
        "source_system",
        "load_timestamp"
    )
    .withColumn(
        "silver_load_timestamp",
        current_timestamp()
    )
    .withColumn(
        "pipeline_name",
        lit("bronze_to_silver")
    )
    .withColumn(
        "record_status",
        lit("ACTIVE")
    )
)

In [0]:
valid_providers = silver_providers.filter(
    col("provider_id").isNotNull()
)

In [0]:
(
    valid_providers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{env_catalog}.silver.providers"
    )
)